In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e10/sample_submission.csv
/kaggle/input/playground-series-s5e10/train.csv
/kaggle/input/playground-series-s5e10/test.csv
/kaggle/input/s5e10-oof-predictions/ensemble_oof_ridge_29models_rmse0.055787_20251028_203224.npy
/kaggle/input/s5e10-oof-predictions/ensemble_oof_greedy_25of97_rmse0.055790_20251026_232051.npy
/kaggle/input/s5e10-oof-predictions/ensemble_test_greedy_25of97_rmse0.055790_20251026_232051.npy
/kaggle/input/s5e10-oof-predictions/ensemble_oof_greedy_21of96_rmse0.055791_20251024_071420.npy
/kaggle/input/s5e10-oof-predictions/ensemble_test_greedy_26of97_rmse0.055789_20251026_102923.npy
/kaggle/input/s5e10-oof-predictions/ensemble_oof_greedy_26of97_rmse0.055789_20251026_102923.npy
/kaggle/input/s5e10-oof-predictions/ensemble_test_ridge_29models_rmse0.055787_20251028_203224.npy
/kaggle/input/s5e10-oof-predictions/ensemble_test_greedy_21of96_rmse0.055791_20251024_071420.npy


In [2]:
import numpy as np
import pandas as pd
from glob import glob
from sklearn.metrics import mean_squared_error

# Load data
train = pd.read_csv("/kaggle/input/playground-series-s5e10/train.csv")
test = pd.read_csv("/kaggle/input/playground-series-s5e10/test.csv")

# Auto-detect target column (last column that's not 'id')
target_col = [col for col in train.columns if col != 'id'][-1]
print(f"Using target column: {target_col}")
y_true = train[target_col].values

# Load all OOF and test predictions
oof_files = sorted(glob("/kaggle/input/s5e10-oof-predictions/*oof*.npy"))
test_files = sorted(glob("/kaggle/input/s5e10-oof-predictions/*test*.npy"))

print(f"Found {len(oof_files)} OOF files and {len(test_files)} test files")

oof_preds = [np.load(f) for f in oof_files]
test_preds = [np.load(f) for f in test_files]

# Simple Average Ensemble
oof_avg = np.mean(oof_preds, axis=0)
test_avg = np.mean(test_preds, axis=0)

# Calculate RMSE CV score
rmse_cv = np.sqrt(mean_squared_error(y_true, oof_avg))
print(f"Simple Average RMSE CV: {rmse_cv:.6f}")

# Save OOF and test predictions
np.save('oof_predictions.npy', oof_avg)
np.save('test_predictions.npy', test_avg)
print("✓ OOF and test predictions saved!")

# Create submission
submission = pd.DataFrame({'id': test['id'], target_col: test_avg})
submission.to_csv('submission.csv', index=False)
print("✓ Submission saved!")

Using target column: accident_risk
Found 4 OOF files and 4 test files
Simple Average RMSE CV: 0.055788
✓ OOF and test predictions saved!
✓ Submission saved!
